# Offline evaluation on `shop.history_sessions`

Before putting a new recommender live, estimate offline whether it beats the baseline.

The log was produced by the **old system** (best sellers plus a bit of randomness), so it only tells you what happened with the products *it* showed. Two estimates:

1. **Hit rate**: of the impressions where the shopper clicked, how often is the clicked product in your top 10? Easy, but biased towards popular products (the old system showed them most).
2. **Inverse propensity scoring (IPS)** of the click rate in **slot 1**: keep only the impressions where your first product equals the one the old system put in slot 1, and weight each by `1 / propensity`. This corrects the popularity bias of the log for that slot. SNIPS (self-normalised IPS) has lower variance.

We hold out the **last 7 days** of the history: the policies are built only from data before the cut and evaluated after it.

In [24]:
import os
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import requests

for line in Path("../.env").read_text().splitlines() if Path("../.env").exists() else []:
    key, _, value = line.partition("=")
    os.environ.setdefault(key.strip(), value.strip())
API = os.environ.get("RAWTREE_API_URL", "https://api.rawtree.com")
http = requests.Session()
http.headers["Authorization"] = f"Bearer {os.environ['RAWTREE_API_KEY']}"

def sql(q):
    r = http.post(f"{API}/v1/query", json={"sql": q}, timeout=600)
    r.raise_for_status()
    return pd.DataFrame(r.json()["data"])

## Data

In [25]:
last_day = sql("SELECT toDate(max(ts::DateTime64(3))) AS d FROM shop.history_sessions").d[0]
CUT = str((pd.Timestamp(last_day) - pd.Timedelta(days=7)).date())   # 'YYYY-MM-DD'
print("train < ", CUT, "<= test")

# Columns are Dynamic (schemaless): filters work as is; joins and some aggregates need casts like ::String
test = sql(f"""
    SELECT request_id, ts, user_id, context, recent_views, shown, propensity, clicked, purchased
    FROM shop.history_sessions WHERE ts >= '{CUT}' ORDER BY rand() LIMIT 100000""")
test["clicked"] = test.clicked.replace("", None)
print(len(test), "test impressions, observed CTR", round(test.clicked.notna().mean(), 4))
test.head(3)

train <  2026-11-07 <= test
100000 test impressions, observed CTR 0.0376


,request_id,ts,user_id,context,recent_views,shown,propensity,clicked,purchased
0,r_059606217823af39,2026-11-07 02:57:07.094000000,u_663e88ca,"{'device': 'ios', 'page': 'category', 'region'...","[{'product_id': 'p_7f4cc0', 'ts': '2026-11-07 ...","[p_5639e2, p_cf5e6d, p_52e7e7, p_8f64c9, p_022...","[0.029527, 0.044663, 0.06966, 0.029863, 0.0146...",NaN,NaN
1,r_088321b7ff7b0886,2026-11-11 01:32:56.221000000,u_47db6440,"{'device': 'web', 'page': 'category', 'region'...","[{'product_id': 'p_0a928a', 'ts': '2026-11-11 ...","[p_c46e14, p_e5ac6e, p_adc3e6, p_7aa466, p_283...","[0.030684, 0.010215, 0.28, 0.012825, 0.004364,...",NaN,NaN
2,r_b5a401cbcb4957f9,2026-11-09 16:08:32.184000000,u_288bb2ed,"{'device': 'ios', 'page': 'category', 'region'...","[{'product_id': 'p_09a938', 'ts': '2026-11-09 ...","[p_adc3e6, p_71450d, p_050730, p_09a938, p_d24...","[0.268636, 0.013583, 0.057391, 0.166195, 0.043...",p_adc3e6,NaN


In [26]:
import sys
sys.path.insert(0, ".")
from recommender import best_sellers, recommend   # the same code that runs live

best = best_sellers(until=CUT)

In [27]:
# Co-views computed inside RawTree: for every product, the 20 products most viewed in the same sessions
coviews = sql(f"""
    SELECT a.product_id AS p, b.product_id AS q, count() AS n
    FROM (SELECT session_id::String AS session_id, product_id::String AS product_id FROM shop.history_events WHERE event_type = 'view' AND ts < '{CUT}') AS a
    INNER JOIN (SELECT session_id::String AS session_id, product_id::String AS product_id FROM shop.history_events WHERE event_type = 'view' AND ts < '{CUT}') AS b
      ON a.session_id = b.session_id
    WHERE a.product_id != b.product_id
    GROUP BY p, q ORDER BY n DESC LIMIT 20 BY p""")
covisit = coviews.groupby("p").q.apply(list).to_dict()
print(len(covisit), "products with co-views")

5000 products with co-views


In [28]:
# c04: importar nuestro recommender con artefactos de corte
import os, sys
os.environ["MODEL_SUFFIX"] = "_cut"
sys.path.insert(0, ".")
sys.modules.pop("recommender", None)     # evita la versión cacheada
import recommender as ours

assert ours.LGB_MODEL is not None, "no se cargó el modelo"

# Ranking de los más vendidos ANTES del corte (el baseline real, sin ver el futuro)
best = sql(f"""
    SELECT product_id::String AS product_id, count() AS n
    FROM shop.history_events
    WHERE event_type = 'purchase' AND ts::DateTime64(3) < toDateTime('{CUT}')
    GROUP BY product_id ORDER BY n DESC LIMIT 200""").product_id.tolist()

Cargando modelo LightGBM y diccionarios de features desde disco...
Cargando modelo LightGBM desde starter/models/...


c:\Users\marcg\TinyBird\venv\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.7.2 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


¡Modelos y features cargados correctamente!


## Policies

A policy maps one request (a row) to an ordered list of 10 product IDs.

In [29]:
# c07: políticas
def baseline(row):
    """Los 10 más vendidos antes del corte (sin filtro de stock offline)."""
    return best[:10]

def coview(row):
    score = Counter()
    for k, v in enumerate(reversed(row.recent_views or [])):
        for rank, q in enumerate(covisit.get(v["product_id"], [])):
            score[q] += 0.7 ** k / (rank + 1)
    items = [q for q, _ in score.most_common(10)]
    return (items + [p for p in baseline(row) if p not in items])[:10]

def lgbm(row):
    return ours.recommend(row.recent_views, best)

POLICIES = {"baseline": baseline, "coview": coview, "lgbm": lgbm}

In [30]:
import recommender, inspect
print(recommender.__file__)
print(hasattr(recommender, "best_sellers"))

c:\Users\marcg\TinyBird\starter\recommender.py
True


## Evaluation

In [31]:
def evaluate(policy, df):
    recs = [policy(r) for r in df.itertuples()]
    clicked = df.clicked.to_numpy()
    has_click = df.clicked.notna().to_numpy()
    hit = np.array([c in r for c, r in zip(clicked, recs)])[has_click].mean()
    first_shown = df.shown.str[0].to_numpy()
    match = np.array([r[0] == s for r, s in zip(recs, first_shown)])
    reward = (clicked == first_shown).astype(float)            # click on the slot-1 product
    w = match / df.propensity.str[0].to_numpy()
    return {"hit_rate@10": hit, "IPS_ctr_slot1": (w * reward).mean(),
            "SNIPS_ctr_slot1": (w * reward).sum() / max(w.sum(), 1e-9), "slot1_matches": int(match.sum())}

old_slot1 = (test.clicked == test.shown.str[0]).mean()
print(f"old system, observed slot-1 CTR: {old_slot1:.4f}")
pd.DataFrame({name: evaluate(p, test) for name, p in POLICIES.items()}).T

old system, observed slot-1 CTR: 0.0114


,hit_rate@10,IPS_ctr_slot1,SNIPS_ctr_slot1,slot1_matches
baseline,0.405685,0.007892,0.053508,3962.0
coview,0.230074,0.009972,0.012969,5428.0
lgbm,0.592721,0.017463,0.034126,12078.0


In [32]:
rng = np.random.default_rng(0)
B = 2000
N = len(test)

def per_row(policy, df):
    """Resultados por fila de una política (una sola pasada por el test)."""
    recs = [policy(r) for r in df.itertuples()]
    shown1 = df.shown.str[0].to_numpy()
    clicked = df.clicked.to_numpy()
    prop1 = df.propensity.str[0].to_numpy().astype(float)
    match = np.array([r[0] == s for r, s in zip(recs, shown1)], dtype=float)
    reward = np.array([c == s for c, s in zip(clicked, shown1)], dtype=float)
    w = match / prop1
    hit = np.array([np.nan if pd.isna(c) else float(c in r)
                    for c, r in zip(clicked, recs)])
    return {"num": w * reward, "w": w, "hit": hit}

R = {name: per_row(p, test) for name, p in POLICIES.items()}

def stats(res, idx):
    """IPS, SNIPS y hit@10 sobre un conjunto de filas (índices)."""
    num, w, hit = res["num"][idx], res["w"][idx], res["hit"][idx]
    return np.array([num.mean(),
                     num.sum() / max(w.sum(), 1e-9),
                     np.nanmean(hit)])

# Los mismos índices bootstrap para todas las políticas: así las diferencias son pareadas
IDX = [rng.integers(0, N, N) for _ in range(B)]
ALL = np.arange(N)
names = ["IPS slot1", "SNIPS slot1", "hit@10"]

def summary(name):
    point = stats(R[name], ALL)
    boot = np.array([stats(R[name], i) for i in IDX])
    lo, hi = np.percentile(boot, [2.5, 97.5], axis=0)
    return pd.DataFrame({"valor": point, "IC95_inf": lo, "IC95_sup": hi}, index=names)

for name in POLICIES:
    print(f"\n== {name} ==")
    print(summary(name).round(4))

def diff(a, b):
    """Diferencia pareada a − b con IC95 (si el IC no incluye 0, la diferencia es significativa)."""
    point = stats(R[a], ALL) - stats(R[b], ALL)
    boot = np.array([stats(R[a], i) - stats(R[b], i) for i in IDX])
    lo, hi = np.percentile(boot, [2.5, 97.5], axis=0)
    return pd.DataFrame({"diff": point, "IC95_inf": lo, "IC95_sup": hi}, index=names)

print("\n== lgbm − baseline ==")
print(diff("lgbm", "baseline").round(4))


== baseline ==
              valor  IC95_inf  IC95_sup
IPS slot1    0.0079    0.0068    0.0090
SNIPS slot1  0.0535    0.0466    0.0605
hit@10       0.4057    0.3897    0.4214

== coview ==
              valor  IC95_inf  IC95_sup
IPS slot1    0.0100    0.0072    0.0133
SNIPS slot1  0.0130    0.0092    0.0176
hit@10       0.2301    0.2177    0.2439

== lgbm ==
              valor  IC95_inf  IC95_sup
IPS slot1    0.0175    0.0156    0.0195
SNIPS slot1  0.0341    0.0305    0.0381
hit@10       0.5927    0.5770    0.6087

== lgbm − baseline ==
               diff  IC95_inf  IC95_sup
IPS slot1    0.0096    0.0080    0.0113
SNIPS slot1 -0.0194   -0.0256   -0.0129
hit@10       0.1870    0.1742    0.2001


## Next steps

- Few `slot1_matches` means a noisy IPS estimate: report it next to the estimate.
- Personal taste: who bought what in `shop.history_events` (matrix factorisation, item-to-item similarity).
- Cold start: new products only have a `description`. Text embeddings can place them next to similar products.
- Context: does price sensitivity change with `device` and `region`?
- Live: the stream changes (trends, price drops, stock-outs). Your `team_NN.feedback` tells you how you are really doing.